# CV01 — EDA, tiền xử lý và phân chia 80/20

Nạp `a10.csv`, kiểm toán dữ liệu, trực quan hóa trend/mùa vụ và chia tuần tự 163 tháng Train / 41 tháng Test.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
from cv01_eda_and_preprocessing import (
    RAW_DATA, load_data, audit_data, plot_eda, split_train_test, save_splits
)


In [ ]:
df = load_data(RAW_DATA)
print(df.head())
print(f'Shape: {df.shape}; index frequency: {df.index.freq}')
assert len(df) == 204
assert str(df.index.freq) == '<MonthBegin>'


## 1. Kiểm toán chất lượng

Kết quả kỳ vọng: 0 missing và 0 duplicate.


In [ ]:
summary = audit_data(df)
summary


## 2. EDA và trả lời câu hỏi Bài 1

- **Trend:** chuỗi có xu hướng tăng rõ rệt; biên độ dao động tăng theo mức chuỗi.
- **Seasonality:** boxplot theo 12 tháng cho thấy các mẫu lặp lại hàng năm, thường cao vào tháng 1 và thấp hơn vào tháng 2–3.
- **Tính dừng:** quan sát cho thấy chuỗi gốc có khả năng không dừng vì mức trung bình thay đổi theo thời gian và phương sai mở rộng; cần kiểm định ADF/sai phân ở CV02.


In [ ]:
plot_eda(df)
from IPython.display import Image, display
display(Image(filename=str(ROOT / 'outputs/figures/cv01_eda_timeseries.png')))
display(Image(filename=str(ROOT / 'outputs/figures/cv01_monthly_seasonality_boxplot.png')))


## 3. Temporal split — không shuffle

Dùng 163 tháng đầu cho Train và 41 tháng cuối cho Test để mô phỏng dự báo tương lai, không làm rò rỉ thông tin thời gian.


In [ ]:
train, test = split_train_test(df, train_ratio=0.80)
save_splits(train, test)
assert len(train) == 163 and len(test) == 41
assert train.index[-1] == pd.Timestamp('2005-01-01')
assert test.index[0] == pd.Timestamp('2005-02-01')
print('Train:', train.index[0].date(), '->', train.index[-1].date(), len(train), 'months')
print('Test: ', test.index[0].date(), '->', test.index[-1].date(), len(test), 'months')
